In [7]:
import os
import re
import time
import shutil
import unicodedata as ud
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import chardet

TXT_DIR = Path("/Volumes/One Touch/8k_file_txt")       # 輸入：存放原始 .txt 檔案的資料夾
COPY_DIR = Path("/Volumes/One Touch/8k_filtered_txt")  # 輸出：篩選結果存放的資料夾
FILE_LIST_PATH = Path("all_txt_files_list.txt")   # 快取清單檔名
KEYWORD_CONFIG_PATH = "/Users/wanghao/Downloads/keywords.xlsx"             # 關鍵字 Excel 路徑

BATCH_SIZE = 50000
MAX_WORKERS = 4

In [3]:
def load_patterns_from_excel(path):
    df = pd.read_excel(path)
    patterns = []
    print("Loading keyword patterns:")
    for _, row in df.iterrows():
        kw = str(row['Keyword']).strip()
        # 處理 regex 特殊字元並將空格轉為 \s+
        safe_kw = re.escape(kw).replace(r"\ ", r"\s+")
        search_type = str(row['Search_Type']).strip().lower()

        if search_type == 'prefix':
            # Prefix: 字首模糊搜尋 (e.g. audit -> audit, auditor)
            regex_str = fr"\b{safe_kw}\w*"
        elif search_type == 'exact':
            # Exact: 精確搜尋 (e.g. bankruptcy -> 僅抓 bankruptcy)
            regex_str = fr"\b{safe_kw}\b"
        else:
            regex_str = safe_kw

        patterns.append({
            "keyword": kw,
            "pattern": re.compile(regex_str, re.IGNORECASE)
        })
        print(f"   [{kw}] ({search_type}) -> Regex: {regex_str}")
    return patterns

def get_all_files(source_dir, list_path):
    files = []
    if list_path.exists():
        print(f"Loading file list from {list_path}...")
        with open(list_path, "r", encoding="utf-8") as f:
            for line in f:
                path = Path(line.strip())
                if path.exists():
                    files.append(path)
    
    if not files:
        print(f"Scanning directory {source_dir} for .txt files (this may take a while)...")
        # 修改：搜尋 *.txt 檔案
        files = list(source_dir.rglob("*.txt"))
        print(f"Saving list to {list_path}...")
        with open(list_path, "w", encoding="utf-8") as f:
            for p in files:
                f.write(str(p) + "\n")
                
    return files

def read_text_best_effort(path: Path) -> str:
    # 維持原本的編碼偵測邏輯，防止亂碼
    try:
        raw = path.read_bytes()
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        enc = chardet.detect(raw).get("encoding") or "latin-1"
        try:
            return raw.decode(enc, errors="replace")
        except:
            return raw.decode("latin-1", errors="replace")

def normalize_text(s: str) -> str:
    # 文字正規化邏輯維持不變
    s = ud.normalize("NFKC", s)
    s = s.lower()
    s = s.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    _ZW_CHARS = r"\u200b\u200c\u200d\u2060\ufeff"
    s = re.sub(f"[{_ZW_CHARS}]", " ", s)
    # 保留英數字與單引號
    s = re.sub(r"[^a-z0-9\s']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def scan_file(file_path, patterns):
    try:
        text_content = read_text_best_effort(file_path)
        clean_text = normalize_text(text_content)

        stats = {}
        total_hits = 0

        for p in patterns:
            matches = p['pattern'].findall(clean_text)
            count = len(matches)
            if count > 0:
                stats[p['keyword']] = count
                total_hits += count

        if total_hits > 0:
            return {
                "filename": file_path.name,
                "file_path": str(file_path),
                "total_hits": total_hits,
                **stats
            }
        return None
    except Exception as e:
        return {"filename": file_path.name, "error": str(e)}

In [9]:
patterns = load_patterns_from_excel(KEYWORD_CONFIG_PATH)

if 'all_files' not in globals() or not all_files:
    all_files = get_all_files(TXT_DIR, FILE_LIST_PATH)

total_available = len(all_files)
print(f"Total files available: {total_available}")

Loading keyword patterns:
   [audit] (prefix) -> Regex: \baudit\w*
   [auditor] (prefix) -> Regex: \bauditor\w*
   [internal control] (exact) -> Regex: \binternal\s+control\b
Scanning directory /Volumes/One Touch/8k_file_txt for .txt files (this may take a while)...
Saving list to all_txt_files_list.txt...
Total files available: 136955


In [11]:
print("-" * 30)
try:
    start_input = input(f"Start Index (default 0): ")
    start_idx = int(start_input) if start_input.strip() else 0
    
    end_input = input(f"End Index (default {total_available}): ")
    end_idx = int(end_input) if end_input.strip() else total_available
except ValueError:
    print("Input error, please enter numbers.")
    start_idx, end_idx = -1, -1

if start_idx < 0 or end_idx > total_available or start_idx >= end_idx:
    print("Invalid range. Please re-run this cell.")
    run_execution = False
else:
    target_files = all_files[start_idx:end_idx]
    target_count = len(target_files)
    print(f"Settings: {start_idx} to {end_idx} (Total {target_count})")
    run_execution = True

------------------------------


Start Index (default 0):  0
End Index (default 136955):  10000


Settings: 0 to 10000 (Total 10000)


In [17]:
if run_execution:
    num_sub_batches = (target_count + BATCH_SIZE - 1) // BATCH_SIZE
    
    print(f"Starting execution. Split into {num_sub_batches} batches...\n")

    for i in range(num_sub_batches):
        local_start = i * BATCH_SIZE
        local_end = min((i + 1) * BATCH_SIZE, target_count)
        
        global_start_idx = start_idx + local_start
        
        # 批次 ID 計算
        batch_id = (global_start_idx // BATCH_SIZE) + 1
        
        output_excel = f"txt_batch_{batch_id}_results.xlsx"
        
        if os.path.exists(output_excel):
            print(f"TXT_Batch {batch_id} ({output_excel}) exists. Skipping.")
            continue
            
        print(f"\n=== Processing Batch {batch_id} (Global Index {global_start_idx} ~ {start_idx + local_end}) ===")
        
        target_batch_dir = COPY_DIR / f"batch_{batch_id}"
        if not target_batch_dir.exists():
            target_batch_dir.mkdir(parents=True, exist_ok=True)

        current_batch_files = target_files[local_start:local_end]
        batch_results = []
        
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_file = {executor.submit(scan_file, f, patterns): f for f in current_batch_files}
            
            processed = 0
            copied_count = 0
            
            for future in as_completed(future_to_file):
                res = future.result()
                processed += 1
                
                if res and "error" not in res:
                    batch_results.append(res)
                    
                    try:
                        src_path = Path(res["file_path"])
                        dst_path = target_batch_dir / src_path.name
                        shutil.copy2(src_path, dst_path)
                        copied_count += 1
                    except Exception as e:
                        print(f"Copy Error: {src_path} -> {e}")
                
                if processed % 1000 == 0:
                    print(f"\rProgress: {processed}/{len(current_batch_files)} | Copied: {copied_count}", end="")
        
        if batch_results:
            df = pd.DataFrame(batch_results)
            # 調整欄位順序，把 file_path 放最後
            cols = [c for c in df.columns if c != "file_path"] + ["file_path"]
            df = df[cols].fillna(0)
            df.to_excel(output_excel, index=False)
            print(f"\nBatch {batch_id} Done. Report: {output_excel}")
        else:
            pd.DataFrame([{"status": "no_hits"}]).to_excel(output_excel, index=False)
            print(f"\nBatch {batch_id} Done (No hits).")

    print("\nSpecified range completed.")
else:
    print("Please go back to the previous cell to set the range.")

Starting execution. Split into 1 batches...


=== Processing Batch 1 (Global Index 0 ~ 10000) ===
Progress: 10000/10000 | Copied: 2741
Batch 1 Done. Report: txt_batch_1_results.xlsx

Specified range completed.
